Batched Matrix multiplication with einops.einsum


1. introduce

In [ ]:
import numpy
from utils import display_np_arrays_as_images
from einops import rearrange, einsum

display_np_arrays_as_images()

# 调换长和宽 c color不变
rearrange(a, "h w c -> w h c")

# transposition
# 4d -> 3d
# 反过来就是decomposition
rearrange(ims, "b h w c -> (b h) w c")

# 最左边的变量是最重要/显著的， 最右边是变化更快的

2. einops.reduce 压缩维度

In [ ]:
from einops import reduce
# 不用 x.mean(-1)
# 并不显示表达发生了什么
# c 被压缩了（方法是取平均）
reduce(ims, 'b h w c -> b h w', 'mean')
# 或者这样写
ims.mean(axis = 0)

# 还可以有其他处理方法
reduce(ims, "b h w c -> h w", "min")reduce(ims, "b h w c -> h w", "min")



3. stack and concatenate 堆叠和连接


In [ ]:
# that's equivalent to numpy stacking, but written more explicitly
numpy.array_equal(rearrange(x, "b h w c -> h w c b"), numpy.stack(x, axis=3))

# which is equivalent to concatenation
numpy.array_equal(rearrange(x, "b h w c -> h (b w) c"), numpy.concatenate(x, axis=1))

4. 添加或删除维度

In [ ]:
x = rearrange(ims, "b h w c -> b 1 h w 1 c")  # functionality of numpy.expand_dims
print(x.shape)
print(rearrange(x, "b 1 h w 1 c -> b h w c").shape)  # functionality of numpy.squeeze


# compute max in each image individually, then show a difference

x = reduce(ims, "b h w c -> b () () c", "max") - ims
# 作差，得到每个像素距离图最高亮度的差值 b,1, 1, c
rearrange(x, "b h w c -> h (b w) c")

5.重复某一个维度

In [ ]:
# repeat along a new axis. New axis can be placed anywhere
repeat(ims[0], "h w c -> h new_axis w c", new_axis=5).shape

# shortcut
repeat(ims[0], "h w c -> h 5 w c").shape

# repeat along w (existing axis)
repeat(ims[0], "h w c -> h (repeat w) c", repeat=3)

# repeat along two existing axes
repeat(ims[0], "h w c -> (2 h) (2 w) c")

# order of axes matters as usual - you can repeat each element (pixel) 3 times
# by changing order in parenthesis
repeat(ims[0], "h w c -> h (w repeat) c", repeat=3)